# Employee Attrition Simulation

This notebook runs the attrition prediction pipeline.
Switch `DATA_MODE` to select the data source.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# 'synthetic' : use the bundled synthetic data (no application required)
# 'gdrive'    : use JPSED data on Google Drive
# 'local'     : use JPSED data on local filesystem
DATA_MODE = 'synthetic'

GDRIVE_DATA_DIR = '/content/drive/MyDrive/ColabNotebooks/employee-attrition/data'
LOCAL_DATA_DIR  = '/Users/yoshikawadaisuke/My Drive/ColabNotebooks/employee-attrition/data'
REPO_URL        = 'https://github.com/dsk-yshkw/employee-attrition'

In [ ]:
# ── Install / clone repo ───────────────────────────────────────────────────────
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('employee-attrition'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('employee-attrition')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)

    if DATA_MODE == 'gdrive':
        from google.colab import drive
        drive.mount('/content/drive')

In [ ]:
# ── Resolve data directory ─────────────────────────────────────────────────────
if DATA_MODE == 'synthetic':
    DATA_DIR = 'data/synthetic'
elif DATA_MODE == 'gdrive':
    DATA_DIR = GDRIVE_DATA_DIR
else:
    DATA_DIR = LOCAL_DATA_DIR

print(f'DATA_MODE : {DATA_MODE}')
print(f'DATA_DIR  : {DATA_DIR}')

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────────
import pandas as pd
from src.data.loader import DataLoader
from src.data.panel_builder import PanelBuilder

if DATA_MODE == 'synthetic':
    panel = pd.read_csv('data/synthetic/sample_synthetic.csv')
else:
    loader = DataLoader(data_dir=DATA_DIR)
    builder = PanelBuilder(loader)
    panel = builder.build()
    panel = builder.build_transitions(panel)

print(panel.shape)
panel.head()

In [ ]:
# ── Feature engineering ────────────────────────────────────────────────────────
from src.features.demographics import DemographicFeatureBuilder
from src.features.employment import EmploymentFeatureBuilder
from src.features.salary import SalaryFeatureBuilder

demo    = DemographicFeatureBuilder().build(panel)
emp     = EmploymentFeatureBuilder().build(panel)
sal     = SalaryFeatureBuilder().build_from_merged(panel)

X = pd.concat([demo, emp, sal], axis=1)
y = panel['Q46_1'].fillna(0).astype(int)

print(f'Features: {X.shape[1]}  |  Target attrition rate: {y.mean():.3f}')

In [ ]:
# ── Train / test split (time-based) ───────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# ── Train model ────────────────────────────────────────────────────────────────
from src.models.attrition import AttritionModel
from src.models.evaluator import ModelEvaluator

model = AttritionModel(model_type='xgboost')
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)
evaluator = ModelEvaluator()
results = evaluator.evaluate(y_test, y_prob)

print(f"AUC-ROC : {results['auc_roc']:.4f}")
print(f"AUC-PR  : {results['auc_pr']:.4f}")
print(results['classification_report'])

In [ ]:
# ── Visualize ──────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
evaluator.plot_roc(y_test, y_prob, ax=axes[0])
evaluator.plot_pr(y_test, y_prob, ax=axes[1])
plt.tight_layout()
plt.show()